In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler

def get_pwr(f='household_power_consumption.txt'):
    print("Loading")
    d = pd.read_csv(f, sep=';', na_values=['?'], low_memory=False)
    d.dropna(inplace=True)
    print("Cleaned")
    return d

df_p = get_pwr()

Loading
Cleaned


In [2]:
def f_pow(d):
    return d[d['Global_active_power'] > 5]

def f_cur(d):
    return d[(d['Global_intensity'] >= 19) & (d['Global_intensity'] <= 20) & (d['Sub_metering_2'] > d['Sub_metering_3'])]

def f_rnd(d):
    s = d.sample(n=500000, replace=False)
    return s[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()

def f_evn(d):
    s = d[(d['Time'] >= '18:00:00') & (d['Global_active_power'] > 6) & (d['Sub_metering_2'] > d['Sub_metering_1']) & (d['Sub_metering_2'] > d['Sub_metering_3'])]
    h1, h2 = s.iloc[:len(s)//2], s.iloc[len(s)//2:]
    return pd.concat([h1.iloc[::3], h2.iloc[::4]])

def f_scl(d):
    c = ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
    n = pd.DataFrame(MinMaxScaler().fit_transform(d[c]), columns=c)
    s = pd.DataFrame(StandardScaler().fit_transform(d[c]), columns=c)
    print("Scaled")
    return n, s

def f_cor(d, c1, c2):
    p = d[c1].corr(d[c2], method='pearson')
    s = d[c1].corr(d[c2], method='spearman')
    print(f"P: {p:.3f}, S: {s:.3f}")
    return p, s

def f_ohe(d):
    c = d.copy()
    c['Hr'] = c['Time'].str[:2]
    res = pd.get_dummies(c, columns=['Hr'])
    print("OHE done")
    return res

In [3]:
r1 = f_pow(df_p)
r2 = f_cur(df_p)
r3 = f_rnd(df_p)
r4 = f_evn(df_p)
n_df, s_df = f_scl(df_p)
p_val, s_val = f_cor(df_p, 'Global_active_power', 'Global_intensity')
ohe_df = f_ohe(df_p)

print("\n--- Time Profiling ---")
%timeit -n 1 -r 1 f_pow(df_p)
%timeit -n 1 -r 1 f_cur(df_p)
%timeit -n 1 -r 1 f_rnd(df_p)

Scaled
P: 0.999, S: 0.995
OHE done

--- Time Profiling ---
4.33 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
8.14 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
196 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
